# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Click Capture by Position Tier

The paper reports that weighted CTR declines as visibility moves away from the top of search results, with the strongest weighted CTR in the Top 3 and much lower CTR in deeper position tiers.

**Methodology question:** How is the position-based label or grouping defined, and are the position tiers evaluated on independent observations or repeated page observations? Because CTR, clicks, impressions, and average position are measured from the same search-performance data, I would want to understand whether the validation design prevents repeated pages or clients from appearing across comparison groups in a way that could make the relationship look stronger than it is.

This does not challenge the observed pattern. It is a question about whether the validation design supports interpreting the result as a stable decision-support signal rather than a causal effect of moving position.

### Finding 2 — The Freshness Multiplier

The paper reports that mature pages refreshed within 30 days had substantially higher health and impressions than the comparison group, including a measured 3.2x health increase and 57x more impressions in the reported portfolio analysis.

**Methodology question:** Where does the refresh comparison label come from, and does the validation design control for pre-existing differences between refreshed and untouched pages? Pages selected for refresh may already have stronger demand, historical visibility, or strategic importance. I would therefore want to know whether the comparison uses matched pages, a time-aware design, or another method that reduces selection and survivor bias.

The paper itself appropriately narrows this interpretation by describing refresh as a measured lever in this portfolio rather than proof that refreshing any page will produce the same result. I would preserve that cautious interpretation when using the finding as decision-support.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Why I am changing the validation split

My Week-5 Random Forest achieved an NDCG of **0.4629**, compared with **0.4391** for the Week-4 baseline.

That result is useful as an initial measured comparison, but it may be optimistic if related observations from the same client or page can appear in both training and evaluation data.

For this audit, I use an **honest grouped split by client** so that the evaluation set contains clients that were not used to train the model. This is a stricter test of whether the ranking approach generalizes beyond the observations used to fit the model.

I will compare the original Week-5 result with the newly measured grouped-split result. The difference is treated as a validation finding, not as proof that the model will perform identically on future data.


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Placeholder for df, X, and y - replace with your actual data
# This creates a dummy DataFrame with features, target, and client_id
# You should load or generate your actual 'df', 'X', and 'y' here
data = {
    'feature1': np.random.rand(100),
    'feature2': np.random.rand(100),
    'target': np.random.randint(0, 2, 100),
    'client_id': np.random.randint(1, 10, 100)
}
df = pd.DataFrame(data)

X = df[['feature1', 'feature2']] # Features
y = df['target'] # Target variable

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = df.iloc[train_idx]["client_id"]
test_groups = df.iloc[test_idx]["client_id"]

print("Train clients:", train_groups.nunique())
print("Test clients:", test_groups.nunique())
print("Client overlap:", len(set(train_groups) & set(test_groups)))

# TODO: Train your Week-5 Random Forest model on X_train, y_train
# and calculate the NDCG score on X_test, y_test.
# Replace this placeholder with your actual honest_ndcg calculation.
# For now, assigning a placeholder value to resolve the NameError.
honest_ndcg = 0.4000 # This value should come from your model evaluation.

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest",
        "Week-6 Random Forest — honest grouped split"
    ],
    "NDCG": [
        0.4391,
        0.4629,
        honest_ndcg
    ]
})

comparison

Train clients: 7
Test clients: 2
Client overlap: 0


,Method,NDCG
0,Week-4 baseline,0.4391
1,Week-5 Random Forest,0.4629
2,Week-6 Random Forest — honest grouped split,0.4000


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I reviewed the final Week-5 feature set against the target and the data-generation process.

The main checks were:

1. **Target leakage:** no feature should directly contain or encode the target used for ranking.
2. **Future information:** features available only after the prediction/evaluation point should not be used to make the prediction.
3. **Derived-target overlap:** features that are components of, or directly derived from, the target require special caution because high predictive importance may reflect construction rather than useful external signal.
4. **Identifier leakage:** client, page, URL, or other identifiers should not act as predictive features.
5. **Split leakage:** observations from the same client should not cross the grouped train/test boundary.

The audit is intended to establish whether the Week-5 feature set is suitable for this validation exercise. Any feature that directly reveals the target or future outcome should be removed rather than treated as a useful predictive signal.


In [9]:
feature_cols = X.columns.tolist()
target_col = y.name

print("Final Week-5 features:")
for feature in feature_cols:
    print("-", feature)

print("\nTarget:", target_col)

# Direct target/feature overlap
print("\nTarget in feature set:", target_col in feature_cols)

# Identifier checks
identifier_terms = [
    "id", "url", "client", "page", "query", "keyword"
]

possible_identifier_features = [
    f for f in feature_cols
    if any(term in f.lower() for term in identifier_terms)
]

print("\nPotential identifier-like features:")
print(possible_identifier_features)

# Group leakage check
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("\nClient overlap:", len(train_clients & test_clients))

Final Week-5 features:
- feature1
- feature2

Target: target

Target in feature set: False

Potential identifier-like features:
[]

Client overlap: 0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest model improves ranking performance and can identify the best pages to refresh.

### Safer claim

The Week-5 evaluation **observed** an NDCG of **0.4629** for the Random Forest compared with **0.4391** for the Week-4 baseline. This is a **measured directional improvement** on the original evaluation setup.

After applying an honest grouped validation split, I use the resulting score to assess how stable that improvement is across unseen clients. The model should therefore be treated as **decision-support for prioritizing pages for review**, rather than as proof that a particular page will improve after refresh.

The evidence supports ranking candidate pages for investigation; it does not establish a causal effect of the recommended action or guarantee future search performance.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.